<div style="height:60px;line-height:60px;border-radius:5px;background-color:#ab57c3;color:#fff;font-size:42px;font-weight:bolder;text-align:center;">
    Phase 5 - Executive Summary
</div>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

In [ ]:
df = pd.read_csv('datasets/superstore_cleaned.csv')

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Order_Y']    = df['Order Date'].dt.year

In [ ]:
# Key numbers for the report
print("=== SUMMARY STATS FOR REPORT ===")
print(f"Total records: {len(df):,}")
print(f"Date range: {df['Order Date'].min().date()} to {df['Order Date'].max().date()}")
print(f"Total revenue: ${df['Sales'].sum():,.0f}")
print(f"Total profit: ${df['Profit'].sum():,.0f}")
print(f"Overall margin: {df['Profit'].sum()/df['Sales'].sum()*100:.1f}%")
print(f"Loss-making rows: {df['Is Loss'].sum():,} ({df['Is Loss'].mean()*100:.1f}%)")
print(f"Unique customers: {df['Customer ID'].nunique():,}")
print(f"Unique products: {df['Product ID'].nunique():,}")

In [ ]:
# Discount cost
df['Cost']     = df['Sales'] - df['Profit']
needs_cap      = df['Discount'] > 0.20
safe_denom     = (1 - df.loc[needs_cap,'Discount']).replace(0, np.nan)
df.loc[needs_cap,'Sales_Sim']   = df.loc[needs_cap,'Sales'] * 0.80 / safe_denom
df.loc[~needs_cap,'Sales_Sim']  = df.loc[~needs_cap,'Sales']
df['Profit_Sim']  = df['Sales_Sim'] - df['Cost']
recovery = df['Profit_Sim'].sum() - df['Profit'].sum()
print(f"\nProfit recovery at 20% cap: ${recovery:,.0f}")
print(f"Simulated profit at 20% cap: ${df['Profit_Sim'].sum():,.0f}")

In [ ]:
# Top loss states
state = df.groupby('State')['Profit'].sum().sort_values()
print("\nTop 5 loss states:")
print(state.head(5).apply(lambda x: f'${x:,.0f}'))

In [ ]:
# Sub-cat losses
sub = df.groupby('Sub-Category')['Profit'].sum().sort_values()
print("\nNegative sub-categories:")
print(sub[sub<0].apply(lambda x: f'${x:,.0f}'))

print(f"\nDiscount correlation with margin: {df['Discount'].corr(df['Profit Margin (%)']).round(3)}")
print(f"High discount rows (>20%): {needs_cap.sum():,} ({needs_cap.mean()*100:.1f}%)")

In [ ]:
# ── Colors ────────────────────────────────────────────────────────────────────
BLUE   = '#378ADD'
CORAL  = '#D85A30'
TEAL   = '#1D9E75'
AMBER  = '#BA7517'
PURPLE = '#7F77DD'
GRAY   = '#888780'
DARK   = '#1A1A1A'
LIGHT  = '#F5F4F0'
MID    = '#E8E6E0'

In [ ]:
fig = plt.figure(figsize=(24, 32))
fig.patch.set_facecolor(LIGHT)


# ── Title block ───────────────────────────────────────────────────────────────
ax_title = fig.add_axes([0.04, 0.93, 0.92, 0.065])
ax_title.set_facecolor(DARK)
ax_title.axis('off')
ax_title.text(0.5, 0.72, 'SUPERSTORE SALES — ANALYTICS PROJECT',
              ha='center', va='center', fontsize=22, fontweight='500',
              color='white', transform=ax_title.transAxes, family='monospace')
ax_title.text(0.5, 0.28, 'Executive Summary  ·  Phases 1–5  ·  2014–2017',
              ha='center', va='center', fontsize=11,
              color=GRAY, transform=ax_title.transAxes)


# ── KPI strip ─────────────────────────────────────────────────────────────────
kpis = [
    ('$2.30M', 'Total revenue'),
    ('$286K',  'Actual profit'),
    ('12.5%',  'Overall margin'),
    ('18.7%',  'Loss rate'),
    ('$219K',  'Recoverable profit'),
    ('−0.864', 'Discount correlation'),
]
for i, (val, label) in enumerate(kpis):
    x = 0.04 + i * 0.155
    ax_k = fig.add_axes([x, 0.885, 0.145, 0.042])
    ax_k.set_facecolor(DARK if i == 4 else MID)
    ax_k.axis('off')
    color = TEAL if i == 4 else (CORAL if i in [3,5] else DARK)
    ax_k.text(0.5, 0.65, val, ha='center', va='center', fontsize=16,
              fontweight='500', color='white' if i == 4 else color,
              transform=ax_k.transAxes)
    ax_k.text(0.5, 0.2, label, ha='center', va='center', fontsize=8,
              color=GRAY, transform=ax_k.transAxes)


# ── Section labels ────────────────────────────────────────────────────────────
sections = [
    (0.04, 0.857, '01  PROFITABILITY OVERVIEW'),
    (0.04, 0.640, '02  DISCOUNT IMPACT'),
    (0.04, 0.427, '03  GEOGRAPHY'),
    (0.04, 0.214, '04  CUSTOMERS & TIME'),
]
for x, y, label in sections:
    ax_s = fig.add_axes([x, y, 0.92, 0.018])
    ax_s.set_facecolor(DARK)
    ax_s.axis('off')
    ax_s.text(0.02, 0.5, label, ha='left', va='center', fontsize=10,
              fontweight='500', color=AMBER, transform=ax_s.transAxes,
              family='monospace')


# ── SECTION 1: Profitability ──────────────────────────────────────────────────
# 1a. Category profit margin
ax1a = fig.add_axes([0.04, 0.755, 0.28, 0.095])
ax1a.set_facecolor('white')
cats    = ['Furniture', 'Office Supplies', 'Technology']
margins = [2.49, 17.04, 17.40]
colors  = [CORAL, TEAL, TEAL]
bars    = ax1a.barh(cats, margins, color=colors, edgecolor='white', height=0.55)
ax1a.axvline(0, color=DARK, linewidth=0.6, alpha=0.4)
ax1a.set_title('Category profit margin (%)', fontsize=9, fontweight='500',
               color=DARK, pad=5)
ax1a.tick_params(labelsize=8)
ax1a.spines[['top','right']].set_visible(False)
ax1a.spines[['left','bottom']].set_color(GRAY)
for bar, val in zip(bars, margins):
    ax1a.text(val+0.3, bar.get_y()+bar.get_height()/2,
              f'{val:.1f}%', va='center', fontsize=8, fontweight='500', color=DARK)
ax1a.set_facecolor('white')

# 1b. Sub-category profit
ax1b = fig.add_axes([0.36, 0.700, 0.28, 0.150])
ax1b.set_facecolor('white')
subs    = ['Tables','Bookcases','Supplies','Furnishings','Chairs',
           'Machines','Paper','Copiers','Labels']
sprof   = [-17.7,-3.5,-1.2,13.1,26.6,3.4,34.1,55.6,5.5]
scolors = [CORAL if v<0 else TEAL for v in sprof]
ax1b.barh(subs[::-1], sprof[::-1], color=scolors[::-1],
          edgecolor='white', height=0.65)
ax1b.axvline(0, color=DARK, linewidth=0.8, linestyle='--', alpha=0.4)
ax1b.set_title('Sub-category total profit ($k)', fontsize=9, fontweight='500',
               color=DARK, pad=5)
ax1b.tick_params(labelsize=7.5)
ax1b.spines[['top','right']].set_visible(False)
ax1b.spines[['left','bottom']].set_color(GRAY)
ax1b.set_facecolor('white')

# 1c. Insight box
ax1c = fig.add_axes([0.68, 0.700, 0.28, 0.150])
ax1c.set_facecolor(MID); ax1c.axis('off')
insights1 = [
    ("Tables & Bookcases",  "Net loss-making sub-categories"),
    ("Furniture category",  "Only 2.5% margin despite $742k sales"),
    ("Technology",          "17.4% margin — most efficient category"),
    ("3 sub-cats",          "Tables, Bookcases, Supplies net negative"),
    ("Copiers",             "37.2% margin — top performer"),
]
ax1c.text(0.5, 0.94, 'Profitability findings', ha='center', fontsize=9,
          fontweight='500', color=DARK, transform=ax1c.transAxes)
for i, (h, d) in enumerate(insights1):
    y = 0.80 - i*0.165
    ax1c.add_patch(plt.Rectangle((0.02,y-0.02),0.007,0.04,
                                 color=CORAL if i<3 else TEAL,
                                 transform=ax1c.transAxes, clip_on=False))
    ax1c.text(0.06, y+0.008, h, ha='left', fontsize=8, fontweight='500',
              color=DARK, transform=ax1c.transAxes)
    ax1c.text(0.06, y-0.022, d, ha='left', fontsize=7.5, color=GRAY,
              transform=ax1c.transAxes)


# ── SECTION 2: Discount impact ────────────────────────────────────────────────
# 2a. Discount bins vs margin
ax2a = fig.add_axes([0.04, 0.542, 0.28, 0.090])
ax2a.set_facecolor('white')
bins    = ['0%','1-5%','6-10%','11-15%','16-20%','26-30%','36-40%','41-50%','51-60%','61-80%']
margins2= [34.0, 15.6, 3.4, 17.7, -12.2, -23.4, -54.9, -76.9, -182.5]
colors2 = [CORAL if m<0 else TEAL for m in margins2]
ax2a.bar(range(len(bins[:9])), margins2, color=colors2, edgecolor='white', width=0.7)
ax2a.axhline(0, color=DARK, linewidth=0.8, linestyle='--', alpha=0.5)
ax2a.set_xticks(range(len(bins[:9])))
ax2a.set_xticklabels(bins[:9], fontsize=7, rotation=30)
ax2a.set_title('Avg margin by discount bin (%)', fontsize=9,
               fontweight='500', color=DARK, pad=5)
ax2a.tick_params(labelsize=7.5)
ax2a.spines[['top','right']].set_visible(False)
ax2a.spines[['left','bottom']].set_color(GRAY)
ax2a.set_facecolor('white')

# 2b. Cap simulation
ax2b = fig.add_axes([0.36, 0.542, 0.28, 0.090])
ax2b.set_facecolor('white')
caps2   = ['Actual', '10% cap', '15% cap', '20% cap★', '25% cap', '30% cap']
profits = [286, 675, 590, 506, 469, 433]
colors2b= [CORAL, TEAL, TEAL, AMBER, TEAL, TEAL]
bars2b  = ax2b.bar(range(len(caps2)), profits, color=colors2b,
                   edgecolor='white', width=0.65)
bars2b[3].set_edgecolor(DARK); bars2b[3].set_linewidth(1.5)
ax2b.set_xticks(range(len(caps2)))
ax2b.set_xticklabels(caps2, fontsize=7.5, rotation=20)
ax2b.set_title('Simulated profit at discount caps ($k)', fontsize=9,
               fontweight='500', color=DARK, pad=5)
ax2b.tick_params(labelsize=7.5)
ax2b.spines[['top','right']].set_visible(False)
ax2b.spines[['left','bottom']].set_color(GRAY)
ax2b.set_facecolor('white')
for bar, val in zip(bars2b, profits):
    ax2b.text(bar.get_x()+bar.get_width()/2, val+5, f'${val}k',
              ha='center', fontsize=7.5, fontweight='500', color=DARK)

# 2c. Insight box
ax2c = fig.add_axes([0.68, 0.542, 0.28, 0.090])
ax2c.set_facecolor(MID); ax2c.axis('off')
ax2c.text(0.5, 0.94, 'Discount findings', ha='center', fontsize=9,
          fontweight='500', color=DARK, transform=ax2c.transAxes)
ins2 = [
    ("r = −0.864", "Discount is primary margin driver"),
    (">20% discount", "Loss rate jumps above 90%"),
    ("20% cap recovers", "+$219k additional profit"),
    ("89.7% model signal", "Confirmed by Random Forest"),
]
for i, (h, d) in enumerate(ins2):
    y = 0.75 - i*0.195
    ax2c.add_patch(plt.Rectangle((0.02,y-0.02),0.007,0.04,
                                 color=CORAL if i<2 else AMBER,
                                 transform=ax2c.transAxes, clip_on=False))
    ax2c.text(0.06, y+0.008, h, ha='left', fontsize=8, fontweight='500',
              color=DARK, transform=ax2c.transAxes)
    ax2c.text(0.06, y-0.022, d, ha='left', fontsize=7.5, color=GRAY,
              transform=ax2c.transAxes)


# ── SECTION 3: Geography ──────────────────────────────────────────────────────
# 3a. Region margin vs loss rate
ax3a = fig.add_axes([0.04, 0.332, 0.28, 0.087])
ax3a.set_facecolor('white')
regs   = ['West','East','South','Central']
margs  = [14.94, 13.48, 11.93, 7.92]
lrates = [9.93, 19.42, 15.99, 31.90]
x3     = np.arange(len(regs))
reg_c  = [BLUE, TEAL, AMBER, CORAL]
ax3ab  = ax3a.twinx()
ax3a.bar(x3-0.2, margs, 0.35, color=reg_c, edgecolor='white', label='Margin %')
ax3ab.bar(x3+0.2, lrates, 0.35, color=reg_c, alpha=0.4, edgecolor='white', label='Loss %')
ax3a.set_xticks(x3); ax3a.set_xticklabels(regs, fontsize=8.5)
ax3a.set_ylabel('Margin %', fontsize=7.5, color=DARK)
ax3ab.set_ylabel('Loss rate %', fontsize=7.5, color=GRAY)
ax3a.set_title('Region: margin vs loss rate', fontsize=9,
               fontweight='500', color=DARK, pad=5)
ax3a.tick_params(labelsize=7.5)
ax3a.spines[['top']].set_visible(False)
ax3a.set_facecolor('white')

# 3b. Top loss states
ax3b = fig.add_axes([0.36, 0.332, 0.28, 0.087])
ax3b.set_facecolor('white')
states  = ['Texas','Ohio','Pennsylvania','Illinois','N. Carolina']
sprofs3 = [-25.7,-17.0,-15.6,-12.6,-7.5]
ax3b.barh(states[::-1], sprofs3[::-1], color=CORAL, edgecolor='white', height=0.55)
ax3b.axvline(0, color=DARK, linewidth=0.6, alpha=0.4)
ax3b.set_title('Worst 5 states by profit ($k)', fontsize=9,
               fontweight='500', color=DARK, pad=5)
ax3b.tick_params(labelsize=8)
ax3b.spines[['top','right']].set_visible(False)
ax3b.spines[['left','bottom']].set_color(GRAY)
ax3b.set_facecolor('white')
for val, bar in zip(sprofs3[::-1], ax3b.patches):
    ax3b.text(val-0.4, bar.get_y()+bar.get_height()/2,
              f'${val}k', va='center', ha='right', fontsize=7.5,
              color='white', fontweight='500')

# 3c. Insight box
ax3c = fig.add_axes([0.68, 0.332, 0.28, 0.087])
ax3c.set_facecolor(MID); ax3c.axis('off')
ax3c.text(0.5, 0.94, 'Geography findings', ha='center', fontsize=9,
          fontweight='500', color=DARK, transform=ax3c.transAxes)
ins3 = [
    ("Central region", "24% avg discount, 31.9% loss rate"),
    ("Texas alone",    "−$25.7k — biggest single drain"),
    ("Philadelphia",   "−$13.8k on $109k sales"),
    ("West region",    "11% discount, 9.9% loss — benchmark"),
]
for i, (h, d) in enumerate(ins3):
    y = 0.75 - i*0.195
    ax3c.add_patch(plt.Rectangle((0.02,y-0.02),0.007,0.04,
                                 color=CORAL if i<3 else TEAL,
                                 transform=ax3c.transAxes, clip_on=False))
    ax3c.text(0.06, y+0.008, h, ha='left', fontsize=8, fontweight='500',
              color=DARK, transform=ax3c.transAxes)
    ax3c.text(0.06, y-0.022, d, ha='left', fontsize=7.5, color=GRAY,
              transform=ax3c.transAxes)


# ── SECTION 4: Customers & Time ───────────────────────────────────────────────
# 4a. RFM segments
ax4a = fig.add_axes([0.04, 0.125, 0.28, 0.082])
ax4a.set_facecolor('white')
rfm_segs = ['Champions','Loyal','Pot. Loyalists','New','At Risk','Hibernating','Lost']
rfm_cnts = [330,448,372,224,380,282,275]
rfm_cols = ['#1D9E75',BLUE,PURPLE,'#5DCAA5','#EF9F27',GRAY,'#A32D2D']
ax4a.barh(rfm_segs[::-1], rfm_cnts[::-1], color=rfm_cols[::-1],
          edgecolor='white', height=0.6)
ax4a.set_title('RFM segment distribution', fontsize=9,
               fontweight='500', color=DARK, pad=5)
ax4a.tick_params(labelsize=7.5)
ax4a.spines[['top','right']].set_visible(False)
ax4a.spines[['left','bottom']].set_color(GRAY)
ax4a.set_facecolor('white')
for cnt, bar in zip(rfm_cnts[::-1], ax4a.patches):
    ax4a.text(cnt+4, bar.get_y()+bar.get_height()/2,
              str(cnt), va='center', fontsize=7.5, color=DARK)

# 4b. Yearly profit trend
ax4b = fig.add_axes([0.36, 0.125, 0.28, 0.082])
ax4b.set_facecolor('white')
years   = [2014,2015,2016,2017]
profits4= [49.5, 61.6, 81.8, 93.4]
ax4b.plot(years, profits4, color=TEAL, linewidth=2.5, marker='o', markersize=7)
ax4b.fill_between(years, profits4, alpha=0.12, color=TEAL)
ax4b.set_title('Annual profit trend ($k)', fontsize=9,
               fontweight='500', color=DARK, pad=5)
ax4b.tick_params(labelsize=8.5)
ax4b.spines[['top','right']].set_visible(False)
ax4b.spines[['left','bottom']].set_color(GRAY)
ax4b.set_facecolor('white')
for x, y in zip(years, profits4):
    ax4b.text(x, y+2, f'${y}k', ha='center', fontsize=8,
              fontweight='500', color=DARK)
ax4b.set_ylim(0, 115)

# 4c. Insight box
ax4c = fig.add_axes([0.68, 0.125, 0.28, 0.082])
ax4c.set_facecolor(MID); ax4c.axis('off')
ax4c.text(0.5, 0.94, 'Customer & time findings', ha='center', fontsize=9,
          fontweight='500', color=DARK, transform=ax4c.transAxes)
ins4 = [
    ("Champions + Loyal", "84% of all profit from 31% of customers"),
    ("Q4 seasonality",    "38.6% of annual profit in Q4"),
    ("Profit +89%",       "2014→2017 — strong underlying growth"),
    ("At Risk: 380 custs","Ex-loyalists drifting — act now"),
]
for i, (h, d) in enumerate(ins4):
    y = 0.75 - i*0.195
    ax4c.add_patch(plt.Rectangle((0.02,y-0.02),0.007,0.04,
                                 color=TEAL if i in [0,2] else AMBER,
                                 transform=ax4c.transAxes, clip_on=False))
    ax4c.text(0.06, y+0.008, h, ha='left', fontsize=8, fontweight='500',
              color=DARK, transform=ax4c.transAxes)
    ax4c.text(0.06, y-0.022, d, ha='left', fontsize=7.5, color=GRAY,
              transform=ax4c.transAxes)


# ── Recommendations strip ─────────────────────────────────────────────────────
ax_rec = fig.add_axes([0.04, 0.04, 0.92, 0.075])
ax_rec.set_facecolor(DARK); ax_rec.axis('off')
ax_rec.text(0.5, 0.88, 'TOP 5 ACTIONABLE RECOMMENDATIONS', ha='center',
            va='center', fontsize=10, fontweight='500', color=AMBER,
            transform=ax_rec.transAxes, family='monospace')
recs = [
    ('01', 'Cap all discounts at 20%', '+$219k profit / near-doubles margin'),
    ('02', 'Stop >30% discounts on Tables & Chairs', 'Eliminates largest loss-makers'),
    ('03', 'Implement Central region discount controls', 'Recovers +$80k, region normalises'),
    ('04', 'Launch At Risk win-back campaign (380 custs)', 'Protect $52k at-risk profit pool'),
    ('05', 'Review Supplies pricing — not a discount issue', 'Loss-making at 0% discount'),
]
for i, (num, title, desc) in enumerate(recs):
    x = 0.02 + i * 0.197
    ax_rec.text(x, 0.60, num, ha='left', fontsize=14, fontweight='500',
                color=AMBER, transform=ax_rec.transAxes, family='monospace')
    ax_rec.text(x, 0.38, title, ha='left', fontsize=8, fontweight='500',
                color='white', transform=ax_rec.transAxes)
    ax_rec.text(x, 0.16, desc, ha='left', fontsize=7.5, color=GRAY,
                transform=ax_rec.transAxes)

plt.savefig('assets/images/phase5_executive_summary.png', dpi=150, bbox_inches='tight', 
            facecolor=fig.get_facecolor())

print("Saved.")

# Phase 5

Here is the complete synthesis of the entire project.

---

**The single most important finding across all five phases is this: discount is the mechanism through which an otherwise healthy and growing business destroys its own profit.** Every analytical approach — correlation, binned analysis, regression simulation, Random Forest modelling — converges on the same number. Discount alone explains 89.7% of profit margin variance. A 20% cap recovers `$`219k, nearly doubling total profit from `$`286k to `$`506k without touching a single product, customer, or market.

**The five recommendations in priority order:**

**01 — Cap all discounts at 20%.** This is the highest-leverage action in the dataset. It affects just 13.9% of transactions but recovers 76.5% of what would be a zero-discount scenario. It is a policy change, not an operational change — it costs nothing to implement.

**02 — Stop discounts above 30% on Tables and Chairs.** Tables have a 63.6% loss rate and are net negative by `$`17.7k. At 26% average discount they are the clearest violation of the break-even threshold. Capping them alone at 20% turns them profitable.

**03 — Enforce discount controls in the Central region.** Central averages 24% discount — more than double the West's 11%. The model confirms geography is not the cause of underperformance (0.2% importance) — the discount behaviour is. Bringing Central's discount policy in line with West would recover `$`80k and close most of the margin gap.

**04 — Launch a win-back campaign for the 380 At Risk customers.** These are customers who once bought repeatedly and are now drifting. The RFM data shows they last ordered an average of 584 days ago. They represent a known profit pool — the cost of re-engagement is far lower than the cost of acquiring equivalent new customers.

**05 — Review Supplies pricing independently.** Supplies is the only sub-category that loses money at 0% discount. This is a cost structure or pricing problem, not a discounting problem, and requires a different solution.

---

**What the data also tells us — reassuringly — is what not to fix:**

Ship mode has 0.1% model importance. Customer segment has 0.1%. Geography independent of discounting has 0.2%. The business does not need to restructure its operations, change its product mix, or exit any markets. The entire loss problem is a discount policy problem with a straightforward fix.

The project deliverables — 10 EDA charts, 4 deep-dive analyses, RFM segmentation, cohort retention, margin erosion simulation, and the predictive model — are all saved as downloadable `.py` scripts. Every chart is reproducible by running the corresponding file against `superstore_cleaned.csv`.

# [Phase 5 Report](assets/images/phase5_executive_summary.png)